# 04 — 논문과 코드 나란히 걷기

**무엇을 확인하는가**

1. 앵커에 걸린 코드 원문을 직접 읽는다
2. `REP-001` — BatteryLife 와 BatteryMFormer 의 동명 파일이 정말 다 다른가
3. 논문 슬롯을 채울 때 무엇을 적어야 하는지 확인한다

논문 PDF 는 저장소에 없습니다 (`python run.py papers` 로 각자 받습니다).
이 노트북은 **코드 쪽** 을 정확히 보여주는 데까지가 역할이고, 논문 쪽은
사람이 읽어 `registry.yaml` 의 `paper` 슬롯에 적습니다.

## 0. 부트스트랩

In [ ]:
import sys
from pathlib import Path

# 노트북에서 저장소 루트를 import 경로에 넣습니다.
REPO = Path.cwd()
while not (REPO / "run.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("저장소 루트:", REPO)

from verify import FINDINGS, load_yaml, read_text
from verify import anchors as anchors_mod

## 1. 앵커 원문 읽기

행 번호가 아니라 원문이 기준입니다. 아래는 등록된 스니펫 그대로입니다.

In [ ]:
document = load_yaml(FINDINGS / "anchors.yaml")
for anchor in document["anchors"]:
    print("=" * 74)
    print(f"{anchor['id']}   {anchor['path']}:{anchor['line']}-{anchor['line_end']}")
    print(f"무엇: {anchor['what']}")
    print(f"레코드: {anchor.get('records')}")
    print("-" * 74)
    print(read_text(FINDINGS / "snippets" / anchor["snippet_file"]))

## 2. 앵커가 아직 유효한가

In [ ]:
rows = anchors_mod.check()
print(anchors_mod.format_report(rows))

## 3. REP-001 — 동명 파일 비교

두 저장소에 이름이 같은 파일이 있습니다. **하나라도 같다고 가정하면 안
됩니다.** 같은 이름이라 섞어 쓰기 쉬운 것이 위험한 지점입니다.

In [ ]:
import hashlib

BLIFE = REPO / "upstream" / "BatteryLife"
BMF = REPO / "upstream" / "BatteryMFormer"

if not BMF.exists() or not any(BMF.iterdir()):
    print("BatteryMFormer 가 비어 있습니다. git submodule update --init")
else:
    blife_files = {
        p.relative_to(BLIFE).as_posix(): p
        for p in BLIFE.rglob("*.py")
    }
    bmf_files = {
        p.relative_to(BMF).as_posix(): p
        for p in BMF.rglob("*.py")
    }
    common = sorted(set(blife_files) & set(bmf_files))

    same, differ = [], []
    for rel in common:
        a = hashlib.sha256(blife_files[rel].read_bytes()).hexdigest()
        b = hashlib.sha256(bmf_files[rel].read_bytes()).hexdigest()
        (same if a == b else differ).append(rel)

    print(f"동명 파일 {len(common)}개 — 같음 {len(same)} / 다름 {len(differ)}\n")
    for rel in differ:
        size_a = blife_files[rel].stat().st_size
        size_b = bmf_files[rel].stat().st_size
        print(f"  다름  {rel:44} {size_a:>8} vs {size_b:>8} bytes")
    for rel in same:
        print(f"  같음  {rel}")

    print("\nREP-001 의 code 슬롯 value 에 이 목록을 적으십시오.")
    print("하나라도 '같음' 이면 씨앗 레코드의 서술(13개 전부 상이)이 틀린 것입니다.")
    print("틀렸으면 틀렸다고 고치십시오 — 문서가 아니라 측정이 기준입니다.")

## 4. 논문 슬롯을 채울 때

`registry.yaml` 을 직접 여십시오. 채울 때 지킬 것 셋:

1. **`확인` 이면 `locus` 를 반드시 적습니다.** `§4.2 p.5`, `Table 1`,
   `부록 A.2 p.14` 처럼.
2. **`부재확인` · `조사했으나불명` 이면 `searched` 에 어디를 봤는지 적습니다.**
   "논문 봤음" 은 `searched` 가 아닙니다. 다음 사람이 같은 곳을 다시 뒤지지
   않을 만큼 적으십시오.
3. **`verdict` 를 적지 마십시오.** `python run.py claims` 가 유도합니다.

`미조사` 를 `조사했으나불명` 으로 말하지 마십시오. 안 찾아본 것과 찾아봤는데
없는 것은 다릅니다. 이 구분이 무너지면 저장소 전체가 무의미해집니다.

In [ ]:
from verify import render

summary = render.render_all(write=False)
print(f"레코드 {summary['records']}개")
for problem in summary["problems"]:
    print("  -", problem)
if not summary["problems"]:
    print("기록 요건 위반 없음")

print("\n슬롯을 고친 뒤:  python run.py claims")